# Bathymetry — shaded relief

Render a GEBCO 2020 subset of the Mid-Atlantic Ridge near the Azores as a
**hillshaded** relief map. Hillshading turns a flat depth grid into an
intuitive 3D-looking surface that makes the rift valley and transform faults
pop — exactly how a marine geologist would eyeball seafloor structure.

In [ ]:
import tempfile

import numpy as np
from pyramids.dataset import Dataset, GeoReference
from pyramids.plot import DataStyle

from earthlens.core import EarthLens

In [ ]:
out = tempfile.mkdtemp()
paths = EarthLens(
    data_source='gebco',
    dataset='gebco_2020',
    aoi=[-29.5, 36.2, -27.7, 38.0],
    path=out,
).download(progress_bar=False)

source = Dataset.read_file(paths[0])
arr = np.asarray(source.read_array(), dtype='float32')
arr = arr[0] if arr.ndim == 3 else arr
arr = np.where(arr == 32767, np.nan, arr)
# LightSource needs finite values; fill the (rare) gaps with the deepest value.
filled = np.where(np.isnan(arr), np.nanmin(arr), arr)
print(
    'grid',
    arr.shape,
    '| depth range',
    round(np.nanmin(arr)),
    '..',
    round(np.nanmax(arr)),
    'm',
)

## Hillshade

Light from the NW (`azdeg=315`) at 45 degrees elevation; `dx`/`dy` are the
approximate metres-per-pixel of a 15-arcsec grid so the relief is physically
scaled rather than wildly exaggerated.

In [ ]:
# pyramids shades the relief itself: the same lighting as the old LightSource
# call (azdeg -> azimuth, altdeg -> altitude), but drawn on real coordinates
# rather than the pixel grid the axes had to be switched off to hide.
relief = Dataset.from_array(
    filled,
    no_data_value=np.nan,
    geo_ref=GeoReference(geo=source.geotransform, epsg=source.epsg),
)
relief.plot(
    cmap='gist_earth',
    data_style=DataStyle(
        hillshade={
            'azimuth': 315,
            'altitude': 45,
            'vert_exag': 1.0,
            'dx': 460,
            'dy': 460,
            'blend_mode': 'soft',
        }
    ),
    title='GEBCO 2020 shaded relief — Mid-Atlantic Ridge (Azores)',
)